# Backtest 005g — Regime as position sizing, not as an entry gate

`backtest_005f` uses the GKYZ regime as a **hard entry gate**: an unfavourable
state blocks the trade outright. `backtest_005e` used the same regime as a
**size multiplier** instead. This notebook takes 005f's strategy and runs every
regime variant the 005e way — each unfavourable condition **halves the stake**
rather than cancelling the entry.

`regime_gkyz_x_bond_yield.ipynb` supplies the two candidate states being tested:
`normalize=False` GKYZ (which separated forward volatility about twice as well as
the normalised version, +7.0pp vs +3.7pp) and a near-orthogonal bond-yield trend.

## What varies — the size multiplier

| | size = product of these factors |
|--|--|
| **A none** | 1.0 always — the baseline any scheme has to beat |
| **B gkyz-norm** | market GKYZ (`normalize=True`) × per-symbol GKYZ |
| **C gkyz-abs** | market GKYZ (`normalize=False`) × per-symbol GKYZ |
| **D norm+rates** | B × "10Y not rising" |
| **E abs+rates** | C × "10Y not rising" |
| **F rates only** | the yield state alone |

Each leg contributes `1.0` when favourable and `LOW_SIZE` (default 0.5) when not,
and the legs multiply — so two unfavourable conditions give a quarter stake.

## Why sizing makes the comparison cleaner

Under hard gating each variant traded a **different number of symbols**, and
symbols with zero trades dropped out of the mean Sharpe entirely — the survivors
were a selected subset. With sizing, **every variant fires exactly the same
entries on the same 197 symbols**; only the stake changes. Differences in the
results are therefore attributable to sizing alone, not to which symbols happened
to survive the filter.

## Why this notebook re-executes 005f's cells

The TTM signal functions, tuned parameters and loader live in
`backtest_005f_regime_gkyz.ipynb`. Copying them here would fork the strategy
definition and let the two drift apart. Instead the cells are located **by
content** (not index) and exec'd, so this notebook always tests whatever 005f
currently says.

## Read the results with these caveats

* `FIXED_TTM_PARAMS` was tuned by Optuna on this same decade, so the *absolute*
  numbers are optimistic. The **sizing comparison** is still fair — no sizing
  parameter is fitted here, and every variant inherits the same tuned entries.
* The "OOS" slice is the last 40% of the same sample. It is out-of-sample for
  the sizing choice, not for the TTM parameters.
* Mean size differs per variant, so compare **Sharpe** first. A scheme that is
  simply less invested will show a lower total return without being worse.
* A rate state has very few *macro* events even with thousands of symbol-days:
  this decade holds roughly three rate cycles. Treat the yield gate as tested on
  n≈3, not n≈2500.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json, os, pathlib, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path('..').resolve()
# 005f's own import cell only adds the repo root; backend/ is needed as well
# because app/services/indicators/__init__ imports `app.*` absolutely.
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'backend'))

plt.style.use('dark_background')
SURFACE, INK, GRID = '#1a1a19', '#c3c2b7', '#3a3a37'
SLOT = ['#3987e5', '#d95926', '#199e70', '#c98500']    # fixed order, validated on the dark surface
VER_COLOR = {'v1': SLOT[0], 'v2': SLOT[1], 'v3': SLOT[2]}

def style_axes(ax, ylabel=None, title=None):
    ax.grid(True, color=GRID, alpha=0.35, linewidth=0.6)
    ax.set_axisbelow(True)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK, labelsize=9)
    if ylabel: ax.set_ylabel(ylabel, color=INK, fontsize=9)
    if title:  ax.set_title(title, color='white', fontsize=11, loc='left', pad=8)
    return ax

SRC_NB   = ROOT / 'notebooks' / 'backtest_005f_regime_gkyz.ipynb'
CELL_DIR = pathlib.Path('.005g_cells'); CELL_DIR.mkdir(exist_ok=True)
CELLS_SRC = [(''.join(c['source']), c['cell_type'])
             for c in json.loads(SRC_NB.read_text())['cells']]

def run_cell_with(marker, ns):
    '''Exec 005f's first code cell containing `marker`.

    The source is spilled to a real file first: those cells define
    @nb.njit(cache=True) kernels and numba refuses to cache a function whose
    source file it cannot locate on disk.
    '''
    for n, (src, kind) in enumerate(CELLS_SRC):
        if kind == 'code' and marker in src:
            path = CELL_DIR / f'cell_{n:02d}.py'
            path.write_text(src)
            exec(compile(src, str(path), 'exec'), ns)
            return
    raise LookupError(marker)

ns = {'__name__': '__main__'}
t0 = time.time()
for marker in ('import vectorbt as vbt',      # imports
               'def load_stock_data()',        # data (HDF5 cache)
               'def shift_2d(',                # numba helpers
               'def compute_signals(',         # TTM signals + exits
               'def compute_gkyz_regime(',     # regime fns
               'FIXED_TTM_PARAMS = {'):        # tuned params
    run_cell_with(marker, ns)

vbt = ns['vbt']
close, high, low, open_, volume = (ns[k] for k in ('close', 'high', 'low', 'open_', 'volume'))
compute_signals, compute_exits = ns['compute_signals'], ns['compute_exits']
compute_regime_all, compute_gkyz_regime = ns['compute_regime_all'], ns['compute_gkyz_regime']
calculate_gkyz_volatility = ns['calculate_gkyz_volatility']
FIXED_TTM_PARAMS = ns['FIXED_TTM_PARAMS']
print(f'\n005f loaded in {time.time() - t0:.0f}s   panel {close.shape}')

## 1. The three market states

All causal. The absolute-GKYZ threshold is an **expanding** median, so it only
ever uses history to date — a fixed quantile over the whole sample would leak.

In [ ]:
# B — normalised GKYZ hysteresis, exactly as 005f computes it
_, ron_vn_norm = compute_gkyz_regime(open_['VNINDEX'], high['VNINDEX'],
                                     low['VNINDEX'], close['VNINDEX'])

# C — absolute GKYZ vs its own expanding median
gk_abs = pd.Series(
    calculate_gkyz_volatility(open_['VNINDEX'].to_numpy(float), high['VNINDEX'].to_numpy(float),
                              low['VNINDEX'].to_numpy(float), close['VNINDEX'].to_numpy(float),
                              window=21, normalize=False),
    index=close.index)
ron_vn_abs = (gk_abs > gk_abs.expanding(252).median()).fillna(False)

# D/E/F — 10Y yield trend, 21d trailing change with a 10bp deadband
from backend.app.utils.wichart import fetchMacroFrame

YIELD_SYMBOL, YIELD_WINDOW, YIELD_BAND = 'VIETNAM_10Y', 21, 0.10
macro = fetchMacroFrame(log=None)
macro['symbol'] = macro['dim_name'].str.upper()
y10 = (macro[macro['symbol'] == YIELD_SYMBOL].set_index('date')['value']
       .sort_index().reindex(close.index).ffill(limit=5))

def yield_trend_regime(yields, window=YIELD_WINDOW, band=YIELD_BAND):
    dy = yields.diff(window)
    state, out = False, []
    for v in dy.to_numpy():
        if not np.isnan(v):
            if not state and v > band:   state = True
            elif state and v < -band:    state = False
        out.append(state)
    return pd.Series(out, index=yields.index)

rates_up = yield_trend_regime(y10)

pd.DataFrame({
    'state': ['GKYZ normalised risk-ON', 'GKYZ absolute > expanding median', '10Y rising'],
    'share of days': [f'{ron_vn_norm.mean():.0%}', f'{ron_vn_abs.mean():.0%}', f'{rates_up.mean():.0%}'],
    'corr w/ norm': [1.0,
                     round(ron_vn_norm.astype(int).corr(ron_vn_abs.astype(int)), 3),
                     round(ron_vn_norm.astype(int).corr(rates_up.astype(int)), 3)],
})

## 2. Size schemes

In [ ]:
def broadcast(s, cols, index):
    return pd.DataFrame(np.tile(s.to_numpy()[:, None], (1, len(cols))), index=index, columns=cols)


def size_factor(favourable, low=None):
    '''1.0 where the condition is favourable, LOW_SIZE where it is not.

    Same idea as 005e's build_size_matrix, but returned per leg so several legs
    can be multiplied together.
    '''
    return favourable.astype(float).where(favourable, low if low is not None else LOW_SIZE)


def build_size(name, entries, ron_syms, ver):
    '''Per-bar size fractions for one variant — a (bars x symbols) frame.

    The GKYZ legs mirror per version: v3 is mean-reverting and deliberately WANTS
    high volatility, so risk-ON is its favourable state, while v1/v2 want calm.
    The rate leg does not mirror — rising yields are a directional headwind for
    any long-only entry, not a regime a strategy prefers — so every version
    treats "rates not rising" as favourable.
    '''
    cols, idx = entries.columns, entries.index
    want_on = (ver == 'v3')
    full = pd.DataFrame(1.0, index=idx, columns=cols)

    sym_ok  = ron_syms if want_on else ~ron_syms
    rate_ok = ~broadcast(rates_up, cols, idx)
    mkt_ok  = lambda s: (broadcast(s, cols, idx) if want_on else ~broadcast(s, cols, idx))

    gkyz_norm = size_factor(mkt_ok(ron_vn_norm)) * size_factor(sym_ok)
    gkyz_abs  = size_factor(mkt_ok(ron_vn_abs))  * size_factor(sym_ok)
    rates     = size_factor(rate_ok)

    return {
        'A none':       full,
        'B gkyz-norm':  gkyz_norm,
        'C gkyz-abs':   gkyz_abs,
        'D norm+rates': gkyz_norm * rates,
        'E abs+rates':  gkyz_abs * rates,
        'F rates only': rates,
    }[name]


GATES    = ['A none', 'B gkyz-norm', 'C gkyz-abs', 'D norm+rates', 'E abs+rates', 'F rates only']
VERSIONS = ['v1', 'v2', 'v3']
OOS_FRAC = 0.6
LOW_SIZE = 0.5     # stake when a leg is unfavourable; 005e used a half stake

print(f'{len(GATES)} sizing schemes x {len(VERSIONS)} versions x 2 slices = '
      f'{len(GATES) * len(VERSIONS) * 2} portfolios   |   LOW_SIZE={LOW_SIZE}')
print('(actual size levels are printed from the first version in the run below)')

## 3. Run

~1–2 minutes. `entries`/`exits` are computed once per version and reused across
all six gates, so the only per-gate cost is the portfolio simulation.

In [ ]:
rows = []
for ver in VERSIONS:
    p = FIXED_TTM_PARAMS[ver]
    t0 = time.time()
    entries = compute_signals(
        close, high, low, volume,
        bb_window=p['bb_window'], bb_multiplier=p['bb_multiplier'], bb_matype=p.get('bb_matype', 0),
        kc_window=p['kc_window'], kc_multiplier=p['kc_multiplier'], kc_atr_period=p['kc_atr_period'],
        donichan_window=p.get('donichan_window', 10),
        osc_smoothing_period=p.get('osc_smoothing_period', 5),
        entry_version=ver,
        consecutive_neg_threshold=p.get('consecutive_neg_threshold', 7),
        william_vix_period=p.get('william_vix_period', 20),
        use_avwap_filter=True, avwap_window=p.get('avwap_window', 200),
        use_kama_slope=True, kama_period=p.get('kama_period', 10),
        kama_fast=p.get('kama_fast', 2), kama_slow=p.get('kama_slow', 30),
        kama_slope_win=p.get('kama_slope_win', 5),
        flat_threshold_pct=p.get('flat_threshold_pct', 1.0),
    )
    exits, sl_stop = compute_exits(close, high, low,
                                   atr_multiplier=p.get('atr_multiplier', 1.9),
                                   atr_period=p.get('atr_period', 10),
                                   lookback=p.get('low_stop_lookback', 5))
    ron_syms = compute_regime_all(open_, high, low, close, regime_version='v1',
                                  window=p.get('gkyz_window', 21),
                                  upper=p.get('gkyz_upper', 0.8),
                                  lower=p.get('gkyz_lower', 0.2))
    split = int(len(close) * OOS_FRAC)
    if ver == VERSIONS[0]:
        print('  size levels:', {g: [round(float(x), 3) for x in
                                     np.unique(build_size(g, entries, ron_syms, ver).to_numpy())]
                                 for g in GATES})
    for g in GATES:
        size = build_size(g, entries, ron_syms, ver)
        for slice_name, sl in (('full', slice(None)), ('OOS', slice(split, None))):
            pf = vbt.Portfolio.from_signals(
                close=close.iloc[sl], entries=entries.iloc[sl], exits=exits.iloc[sl],
                sl_stop=sl_stop.iloc[sl], size=size.iloc[sl], size_type='percent',
                freq='1d', group_by=['symbol'], cash_sharing=False)
            sh = pf.sharpe_ratio().replace([np.inf, -np.inf], np.nan).dropna()
            # Mean size AT THE ENTRY BARS — the average size over all bars would
            # be dominated by days the strategy is flat and says nothing.
            ent = entries.iloc[sl]
            rows.append({
                'ver': ver, 'gate': g, 'slice': slice_name,
                'mean size': round(float(size.iloc[sl].to_numpy()[ent.to_numpy()].mean()), 3),
                'trades': int(pf.trades.count().sum()),
                'Sharpe mean': round(sh.mean(), 3),
                'Sharpe med': round(sh.median(), 3),
                # Should be a constant 197 in every row: sizing never removes a
                # trade, so no symbol can drop out of the mean. If this column
                # moves, something is silently filtering entries.
                'symbols': len(sh),
                'Sh>1': int((sh > 1).sum()),
                'TotRet mean %': round(pf.total_return().mean() * 100, 1),
                'MaxDD mean %': round(pf.max_drawdown().mean() * 100, 1),
                'win %': round(pf.trades.win_rate().mean() * 100, 1),
            })
    print(f'{ver}: {time.time() - t0:.0f}s')

res = pd.DataFrame(rows)
print()
res[res['slice'] == 'OOS'].drop(columns='slice').to_string(index=False)

In [ ]:
for slice_name in ('full', 'OOS'):
    for ver in VERSIONS:
        print(f'\n===== {ver}  [{slice_name}] =====')
        print(res[(res['ver'] == ver) & (res['slice'] == slice_name)]
              .drop(columns=['ver', 'slice']).to_string(index=False))

## 4. Does the sizing beat flat sizing?

The only comparison that matters. Everything is plotted as **Δ Sharpe against the
flat-size baseline** — a bar below zero means the scheme destroyed value.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
fig.patch.set_facecolor(SURFACE)
gates_x = [g for g in GATES if g != 'A none']
w = 0.26

for ax, slice_name in zip(axes, ('full', 'OOS')):
    ax.set_facecolor(SURFACE)
    for k, ver in enumerate(VERSIONS):
        base = res[(res['ver'] == ver) & (res['slice'] == slice_name) &
                   (res['gate'] == 'A none')]['Sharpe mean'].iloc[0]
        vals = [res[(res['ver'] == ver) & (res['slice'] == slice_name) &
                    (res['gate'] == g)]['Sharpe mean'].iloc[0] - base for g in gates_x]
        xs = np.arange(len(gates_x)) + (k - 1) * w
        ax.bar(xs, vals, width=w * 0.9, color=VER_COLOR[ver], label=ver, zorder=2)
    ax.axhline(0, color=INK, linewidth=1.2, zorder=3)
    ax.set_xticks(range(len(gates_x)), [g.split(' ', 1)[1] for g in gates_x],
                  color=INK, fontsize=9, rotation=12)
    style_axes(ax, 'Δ Sharpe vs flat size' if slice_name == 'full' else None,
               f'{slice_name} — scheme minus flat size')
axes[0].legend(frameon=False, labelcolor=INK, fontsize=9, ncol=3, loc='lower right')
plt.tight_layout()
plt.show()

## 5. Verdict

In [ ]:
oos = res[res['slice'] == 'OOS'].set_index(['ver', 'gate'])

print('OOS mean Sharpe, by version (baseline = flat size):\n')
summary = (res[res['slice'] == 'OOS']
           .pivot(index='gate', columns='ver', values='Sharpe mean')
           .reindex(GATES))
print(summary.to_string(), '\n')

for ver in VERSIONS:
    base = oos.loc[(ver, 'A none'), 'Sharpe mean']
    beat = [g for g in GATES if g != 'A none' and oos.loc[(ver, g), 'Sharpe mean'] > base]
    best = max(GATES, key=lambda g: oos.loc[(ver, g), 'Sharpe mean'])
    print(f'{ver}: flat {base:.3f} | schemes that beat it: {beat or "none"} | best = {best}')

print(f'''\nAnswering the question this was built for:
  * C vs B  — does normalize=False beat the shipped normalised gate, OOS?
      {' '.join(f'{v}: {oos.loc[(v, "C gkyz-abs"), "Sharpe mean"]:.3f} vs {oos.loc[(v, "B gkyz-norm"), "Sharpe mean"]:.3f}' for v in VERSIONS)}
  * Any scheme vs flat size — a scheme that lowers Sharpe is costing you money
    even when it lowers drawdown; check the MaxDD column before deciding that is
    a trade you want.
  * Compare `mean size` alongside `TotRet mean %`. Half the stake roughly halves
    the return arithmetically; only Sharpe and MaxDD say whether the risk was
    well spent.
  * The yield gate is tested against ~3 rate cycles in this sample. Refresh the
    OHLC cache back to 2008 (it starts 2016 today) before trusting it.
  * `symbols` and `trades` should be identical across schemes — sizing changes
    the stake, never whether a trade happens. That is what makes this comparison
    cleaner than the hard-gate version.
''')